In [0]:
%run /Workspace/Users/patelrahul2614@gmail.com/IAP-MainExam/TPCH-Sales&SupplyChain/00_setup

In [0]:
catalog = dbutils.widgets.get("catalog")

In [0]:
# List all tables in samples.tpch
table_names = [row.tableName for row in spark.sql("SHOW TABLES IN samples.tpch").collect()]
# print(f"Found {len(table_names)} tables to ingest to {catalog}.bronze:\n")

streams = []

for table_name in table_names:
    # 1. Create bronze table with CDF enabled if not exists
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {catalog}.bronze.{table_name}
        USING DELTA
        TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
        AS SELECT *,
                  CAST(NULL AS STRING) AS _data_source,
                  CAST(NULL AS TIMESTAMP) AS _ingested_at,
                  CAST(NULL AS STRING) AS _job_run_id
        FROM samples.tpch.{table_name} WHERE 1=0
    """)

    # Ensure CDF is enabled (idempotent for existing tables)
    spark.sql(f"ALTER TABLE {catalog}.bronze.{table_name} SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")

    # 2. Read stream from source with metadata columns
    source_df = add_meta_columns(spark.readStream.table(f"samples.tpch.{table_name}"))

    # 3. Write stream to bronze with checkpoint
    checkpoint_path = f"/Volumes/{catalog}/bronze/checkpoints/{table_name}"
    stream = (source_df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable(f"{catalog}.bronze.{table_name}"))

    streams.append((table_name, stream))
    # print(f"  - {table_name}: stream started")

# print(f"\nWaiting for all streams to complete...")
for table_name, stream in streams:
    stream.awaitTermination()
    # print(f"  - {table_name}: completed")

# print(f"\nAll {len(table_names)} tables ingested to {catalog}.bronze with CDF enabled.")